In [1]:
import pandas as pd
import numpy as np

ruta=r"C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\SLA_Gold.parquet"
sla = pd.read_parquet(ruta)

# Asegurar que la columna de fecha sea datetime (evita comparaciones con strings)
sla["Fecha Finalizacion"] = pd.to_datetime(sla["Fecha Finalizacion"], errors="coerce")

decem = sla[sla["Fecha Finalizacion"] >= pd.Timestamp('2026-01-01')]

# Calcular medias solo para columnas numéricas y de tipo timedeltas (SLA)
num_mean = decem.select_dtypes(include=[np.number]).mean()
td_mean = decem.select_dtypes(include=['timedelta64[ns]']).mean()
mean_sla = pd.concat([num_mean, td_mean])

print("Mean SLA:")
print(mean_sla)

KeyError: 'Fecha Finalizacion'

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Definir tu Top 5 basado en el análisis de Pareto
top_5_soluciones = [
    'CONECTOR DAÑADO',
    'CAMBIO DE VLAN',
    'CONFIGURACION DE EQUIPO',
    'FALLA GENERAL',
    'FALLA LOS'
]

# 2. Filtrar el DataFrame limpio (df_final) para quedarnos solo con el Top 5
df_pareto = df_final[df_final['Solucion Aplicada'].isin(top_5_soluciones)].copy()

# 3. Graficar los histogramas separados para ver la "limpieza" de las curvas
g = sns.FacetGrid(df_pareto, col="Solucion Aplicada", col_wrap=3, height=4, sharex=False, sharey=False)
g.map_dataframe(sns.histplot, x="SLA Resolucion Min", kde=True, bins=30, color='teal')

# Dar formato a los gráficos
g.set_titles(col_template="{col_name}")
g.set_axis_labels("Minutos de Resolución", "Cantidad de Tickets")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np

# Supongamos que df_pareto es tu tabla ya filtrada con el Top 5 de soluciones
# 1. Definir el tamaño del "depósito" o bin (ej. cada 60 minutos = 1 hora)
ancho_bin = 60
max_minutos = int(df_pareto['SLA Resolucion Min'].max())
bins = range(0, max_minutos + ancho_bin, ancho_bin)

# 2. Crear las etiquetas (ej. "0-59", "60-119", etc.)
etiquetas = [f"{i} a {i+ancho_bin-1}" for i in bins[:-1]]

# 3. Asignar cada ticket a su rango correspondiente
df_pareto['Rango_Tiempo_Min'] = pd.cut(df_pareto['SLA Resolucion Min'], bins=bins, labels=etiquetas, right=False)

# 4. AGRUPAR: Esta es la magia que comprime los datos para Power BI
df_gold_distribucion = df_pareto.groupby(['Solucion Aplicada', 'Rango_Tiempo_Min']).size().reset_index(name='Cantidad_Tickets')

# Limpiar rangos donde no hubo ningún ticket para ahorrar aún más peso
df_gold_distribucion = df_gold_distribucion[df_gold_distribucion['Cantidad_Tickets'] > 0]
display(df_gold_distribucion)
# Exportar a Parquet
# df_gold_distribucion.to_parquet("Gold_Distribucion_SLA.parquet")

In [ ]:
import pandas as pd
df = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Ventas_Listado_Gold.parquet')
display(df)

In [ ]:
import pandas as pd
df = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Recaudacion_Gold.parquet')
print(df.columns)
df = df.groupby(['Hora de Pago', 'Vendedor'], as_index=False).size()


display(df)

In [ ]:
import polars as pl
import pandas as pd
df = pd.read_parquet(r'C:\Users\josperez\Downloads\Ventas_Estatus_Gold-Sharepoint.parquet')

mask = (df['Fecha']>=('2026-02-01')) & (df['Fecha']<=('2026-02-28')) &(df['Vendedor'].str.contains("OFI", case=False, na=False))
df_feb=df[mask]
print(df_feb.columns)
df_feb = df_feb.drop_duplicates(subset=['N° Abonado', 'Documento','Fecha','Vendedor','Ciudad','Hora'])
display(df_feb.describe(include='all'))
display(df_feb)

In [ ]:
import pandas as pd
import polars as pl
from pathlib import Path
import duckdb

rutahome = Path.home()
ruta_base = rutahome / 'Documents' / 'A-DataStack' / '01-Proyectos' / '01-Data_PipelinesFibex' / '02_Data_Lake' / 'gold_data'

lf = duckdb.query


In [1]:
import pandas as pd 

df_ventas= pd.read_parquet(r"C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Ventas_Listado_Gold.parquet")
df_estatus= pd.read_parquet(r"C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Ventas_Estatus_Gold.parquet")
df_afluencia = pd.read_parquet(r"C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Afluencia_Gold.parquet")
#display(df_estatus)
display(df_afluencia.info())
fecha = pd.to_datetime('2026-04-01')
mask_ofi_march = (df_ventas['Canal'] == 'OFICINA COMERCIAL') & (df_ventas['Fecha Contrato'] >= fecha)
mask_est_march = (df_estatus['Fecha'] >= fecha)
mask_ventas_march = (df_afluencia['Fecha']>=fecha) & (df_afluencia['Tipo de afluencia'] == 'Ventas')

df_ventas = df_ventas[mask_ofi_march]
df_estatus= df_estatus[mask_est_march]
df_afluencia = df_afluencia[mask_ventas_march]



display(df_ventas.groupby(['Estatus']).agg(Cantidad = ('Estatus','size')))
display(df_estatus.groupby(['Estatus']).agg(Cantidad = ('Estatus','size')))
display(df_afluencia.groupby(['Estatus']).agg(Cantidad = ('Estatus','size')))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1895420 entries, 0 to 1895419
Data columns (total 15 columns):
 #   Column             Dtype                 
---  ------             -----                 
 0   N° Abonado         string                
 1   Documento          string                
 2   Estatus            string                
 3   Fecha              timestamp[ns][pyarrow]
 4   Vendedor           string                
 5   Grupo Afinidad     string                
 6   Nombre Franquicia  string                
 7   Ciudad             string                
 8   Tipo de afluencia  string                
 9   Oficina            object                
 10  Clasificacion      string                
 11  Hora               string                
 12  Mes                string                
 13  Estado_Sede        object                
 14  Tipo_Sede          object                
dtypes: object(3), string(11), timestamp[ns][pyarrow](1)
memory usage: 439.8+ MB


None

,Cantidad
Estatus,
ACTIVO,1732
OBSTRUCCION,16
POR IMPLEMENTACION,55
POR INSTALAR,102
POR VGT,6
SIN FACTIBILIDAD,11


,Cantidad
Estatus,
ACTIVO,1765
ANULADO,16
OBSTRUCCION,15
POR IMPLEMENTACION,46
POR INSTALAR,93
POR VGT,6
RECHAZADO,6
REEMBOLSO ADM,16
REEMBOLSO PAGADOS,21


,Cantidad
Estatus,
ACTIVO,1707
ANULADO,15
OBSTRUCCION,15
POR IMPLEMENTACION,40
POR INSTALAR,108
POR VGT,6
RECHAZADO,6
REEMBOLSO ADM,15
REEMBOLSO PAGADOS,20


In [1]:
import pandas as pd


df = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\bronze_data\Horas_Raw_Bronze.parquet')
df_recaudacion = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\bronze_data\Recaudacion_Raw_Bronze.parquet')
display(df.columns)
display(df)
display(df_recaudacion.columns)
display(df_recaudacion)
df_merge = pd.merge(df_recaudacion, df, on='ID Pago', how='inner')
display(df_merge.head())

Index(['ID Contrato', 'ID Pago', 'N° Abonado', 'Documento', 'Nro Recibo',
       'Fecha', 'Hora de Pago', 'Oficina Cobro', 'Suscripción', 'Source.Name',
       'Fecha_Modificacion_Archivo', 'ID pago', 'Cobrador', 'Forma de Pago',
       'Cliente', 'Total Pago', 'Total Pago Bs', 'Tasa de cambio del día',
       'Usuario de Anulacion'],
      dtype='object')

,ID Contrato,ID Pago,N° Abonado,Documento,Nro Recibo,Fecha,Hora de Pago,Oficina Cobro,Suscripción,Source.Name,Fecha_Modificacion_Archivo,ID pago,Cobrador,Forma de Pago,Cliente,Total Pago,Total Pago Bs,Tasa de cambio del día,Usuario de Anulacion
0,'CONT772929D6D7C85860','6977B69642C0B8654827',1253,8366290,014964,2026-01-26 00:00:00,1899-12-31 14:48:00,OFC - FIBEX ARAGUA-MATURIN,30,Data-HorasPago 30-1 al 2-2.xlsx,2026-02-02 08:53:00.515741,None,None,None,None,None,None,None,None
1,'CONTD365420046F15117','69971B201915B3028843',LCH49718,11382001,525353,2026-02-19 00:00:00,1899-12-31 10:17:00,OFIC-SAN DIEGO-EL RINCON,30,Data-HorasPago 19-2 al 19-2.xlsx,2026-02-19 16:23:10.719427,None,None,None,None,None,None,None,None
2,'CONT1239DC230BC13727','CONT1239DC230BC13727',BQ2870,17229348,183692,2026-02-09 00:00:00,1899-12-31 16:36:00,OFIC-METROPOLIS-BQTO,25,Data-HorasPago 9-2 al 10-2.xlsx,2026-02-10 08:35:12.669482,None,None,None,None,None,None,None,None
3,'CONT690C1B08CE070480','69690CD6B76240852888',VC6426,19060999,063551,2026-01-15 00:00:00,1899-12-31 11:52:00,OFICINA VILLA DE CURA,23.2,Data-HorasPago 30-1 al 2-2.xlsx,2026-02-02 08:53:00.515741,None,None,None,None,None,None,None,None
4,'CONT35B1140825960967','69DCC513C806A0190415',LCH4212,27652247,910880,13/04/2026,06:27 AM,OFI-VIRTUAL,32.5,Data - HorasPago 13-04-2026 al 14-04-2026.xlsx,2026-04-14 16:08:01.185838,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3020800,'CONT06508B1B30A63471','69C15DA3288057485882',LCH55127,10957321,288019,2026-03-23 00:00:00,1899-12-31 11:38:00,OFI-PTO LA CRUZ,30,Data-HorasPago 23-3 al 24-3.xlsx,2026-03-24 10:58:07.044243,None,None,None,None,None,None,None,None
3020801,'CONTD2E8C268F4786717','68922509540525333975',C0971,None,None,2025-08-05 00:00:00,1899-12-31 11:39:00,OFIC GALERIA EL PARAISO,None,Data-HorasPago 01-08 al 27-11.xlsx,2025-12-02 10:03:02.810474,68922509540525301760,SUJAILETH MAIROBIS GARCIA OFIC GALERIAS PARAISO,EFECTIVO DOLLAR - 50.00,None,None,None,None,None
3020802,'CONT6F8975BC1F406007','696F9909D0AA77252940',FC2477,27428501,010713,2026-01-20 00:00:00,1899-12-31 11:02:00,OFC-CUMANACOA,None,Data-HorasPago 20-1 al 20-1.xlsx,2026-01-20 16:53:28.533969,None,None,None,None,None,None,None,None
3020803,'CONT40DDF2975CF01021','6989E7938EE7E6416359',LAG0127,21135779,012861,2026-02-09 00:00:00,1899-12-31 10:00:00,OFC - LAGUNITAS,30,Data-HorasPago 9-2 al 10-2.xlsx,2026-02-10 08:35:12.669482,None,None,None,None,None,None,None,None


Index(['ID Contrato', 'ID Pago', 'N° Abonado', 'Nro Recibo', 'Fecha',
       'Total Pago', 'Total Pago Bs', 'Estatus Pago', 'Forma de Pago',
       'Monto Forma Pago', 'Tasa de cambio del día', 'Banco', 'Referencia',
       'Cobrador', 'Nombre Caja', 'Oficina Cobro', 'Fecha Contrato', 'Estatus',
       'Suscripción', 'Etiqueta', 'Grupo Afinidad', 'Nombre Franquicia',
       'Ciudad', 'Source.Name', 'Fecha_Modificacion_Archivo', 'ID pago'],
      dtype='object')

,ID Contrato,ID Pago,N° Abonado,Nro Recibo,Fecha,Total Pago,Total Pago Bs,Estatus Pago,Forma de Pago,Monto Forma Pago,...,Fecha Contrato,Estatus,Suscripción,Etiqueta,Grupo Afinidad,Nombre Franquicia,Ciudad,Source.Name,Fecha_Modificacion_Archivo,ID pago
0,'CONTF91462D14FB42393','6906207AE6E301052143',V23130,199828,2025-11-01,30,6718.87,PAGADO,TARJETA DE DEBITO,30,...,2023-02-24 00:00:00,ACTIVO,30,None,FIBEX EXPRESS,FIBEX VALENCIA,VALENCIA,Data - Recaudacion 1-11-2025 al 15-11-2025.xlsx,2026-01-14 13:31:59.940364,6906207AE6E301052143
1,'CONTD87627486E485529','69061B8D3D8572666742',GU34478,325426,2025-11-01,10,2240,PAGADO,TARJETA DE DEBITO,10,...,2025-03-17 00:00:00,ACTIVO,20,KARLA REBOLLEDO,FIBEX EXPRESS,FIBEX GUARICO,ELSOMBRERO,Data - Recaudacion 1-11-2025 al 15-11-2025.xlsx,2026-01-14 13:31:59.940364,69061B8D3D8572666742
2,'CONT5855071FF1E85365','690627C515CC77490487',LCH25171,434319,2025-11-01,4.92,1101.89,PAGADO,TARJETA DE DEBITO,4.92,...,2024-05-30 00:00:00,CORTADO,30,None,FIBEX EXPRESS,FIBEX ANZOATEGUI,CUMANA,Data - Recaudacion 1-11-2025 al 15-11-2025.xlsx,2026-01-14 13:31:59.940364,690627C515CC77490487
3,'CONT923CFFCDE3914131','690602BEF13EF0368020',V6820,199589,2025-11-01,20,4479.24,PAGADO,TARJETA DE CREDITO,20,...,2022-05-28 00:00:00,ACTIVO,20,None,FIBEX EXPRESS,FIBEX VALENCIA,VALENCIA,Data - Recaudacion 1-11-2025 al 15-11-2025.xlsx,2026-01-14 13:31:59.940364,690602BEF13EF0368020
4,'CONT6C0A054B90092366','69061987EAF732248447',V4931,199768,2025-11-01,20.88,4676.45,PAGADO,TARJETA DE DEBITO,20.88,...,2022-04-29 00:00:00,ACTIVO,20,None,FIBEX EXPRESS,FIBEX VALENCIA,VALENCIA,Data - Recaudacion 1-11-2025 al 15-11-2025.xlsx,2026-01-14 13:31:59.940364,69061987EAF732248447
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2327726,'CONT9E185B34F1A89935','69B009F17B9315998931',MGTA9736,095975,2026-03-10,25,10499.68,PAGADO,TARJETA DE DEBITO,25,...,2025-02-01 00:00:00,ACTIVO,25,YUSRODRIGUEZ,HOGAR,FIBEX MARGARITA,MARGARITA,Data - Recaudacion 9-3-2026 al 10-3-2026.xlsx,2026-03-10 08:12:53.422155,69B009F17B9315998931
2327727,'CONT1CABC4DCD3688860','69B00A13286B11905410',GU53025,423460,2026-03-10,29.41,12352.89,PAGADO,TARJETA DE DEBITO,29.41,...,2025-11-18 00:00:00,ACTIVO,30,DIANNY MARCANO,HOGAR,FIBEX GUARICO,VALLE DE LA PASCUA,Data - Recaudacion 9-3-2026 al 10-3-2026.xlsx,2026-03-10 08:12:53.422155,69B00A13286B11905410
2327728,'CONT23F867EA84623312','69B00A7F4CAAD1920452',LCH36893,537478,2026-03-10,35,14700,PAGADO,TARJETA DE DEBITO,35,...,2024-10-31 00:00:00,ACTIVO,35,None,HOGAR,FIBEX ANZOATEGUI,CUMANA,Data - Recaudacion 9-3-2026 al 10-3-2026.xlsx,2026-03-10 08:12:53.422155,69B00A7F4CAAD1920452
2327729,'CONTC822D1A0D3041096','69B00A0C7CD036738833',GU50582,423451,2026-03-10,10,4200,PAGADO,TARJETA DE DEBITO,10,...,2025-09-15 00:00:00,ACTIVO,25,DAISY RODRíGUEZ,HOGAR,FIBEX GUARICO,TUCUPIDO,Data - Recaudacion 9-3-2026 al 10-3-2026.xlsx,2026-03-10 08:12:53.422155,69B00A0C7CD036738833


,ID Contrato_x,ID Pago,N° Abonado_x,Nro Recibo_x,Fecha_x,Total Pago_x,Total Pago Bs_x,Estatus Pago,Forma de Pago_x,Monto Forma Pago,...,Source.Name_y,Fecha_Modificacion_Archivo_y,ID pago_y,Cobrador_y,Forma de Pago_y,Cliente,Total Pago_y,Total Pago Bs_y,Tasa de cambio del día_y,Usuario de Anulacion
0,'CONTF91462D14FB42393','6906207AE6E301052143',V23130,199828,2025-11-01,30,6718.87,PAGADO,TARJETA DE DEBITO,30,...,Data-HorasPago 01-08 al 27-11.xlsx,2025-12-02 10:03:02.810474,6906207AE6E301052143,GLEYSI BENAVIDE OFIC TORRE FIBEX VIñEDO,TARJETA DE DEBITO - 30.00,None,None,None,None,None
1,'CONTD87627486E485529','69061B8D3D8572666742',GU34478,325426,2025-11-01,10,2240,PAGADO,TARJETA DE DEBITO,10,...,Data-HorasPago 01-08 al 27-11.xlsx,2025-12-02 10:03:02.810474,69061B8D3D8572666742,AGENTE AUTORIZADO FIBERWAVE YOXIMAR YOXENIA LO...,TARJETA DE DEBITO - 10.00,None,None,None,None,None
2,'CONT5855071FF1E85365','690627C515CC77490487',LCH25171,434319,2025-11-01,4.92,1101.89,PAGADO,TARJETA DE DEBITO,4.92,...,Data-HorasPago 01-08 al 27-11.xlsx,2025-12-02 10:03:02.810474,690627C515CC77490487,AURIMIR DEL VALLE GUEVARA OFIC CUMANA,TARJETA DE DEBITO - 4.92,None,None,None,None,None
3,'CONT923CFFCDE3914131','690602BEF13EF0368020',V6820,199589,2025-11-01,20,4479.24,PAGADO,TARJETA DE CREDITO,20,...,Data-HorasPago 01-08 al 27-11.xlsx,2025-12-02 10:03:02.810474,690602BEF13EF0368020,JHORIANNY ESPINOZA OFIC TORRE FIBEX VIñEDO,TARJETA DE CREDITO - 20.00,None,None,None,None,None
4,'CONT6C0A054B90092366','69061987EAF732248447',V4931,199768,2025-11-01,20.88,4676.45,PAGADO,TARJETA DE DEBITO,20.88,...,Data-HorasPago 01-08 al 27-11.xlsx,2025-12-02 10:03:02.810474,69061987EAF732248447,JHORIANNY ESPINOZA OFIC TORRE FIBEX VIñEDO,TARJETA DE DEBITO - 20.88,None,None,None,None,None


In [ ]:
import pandas as pd
df = pd.read_parquet(r'C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\bronze_data\Atencion_Cliente_Raw_Bronze.parquet')
display(df)


,N° Abonado,Documento,Cliente,Estatus,Saldo,Fecha Llamada,Hora Llamada,Tipo Llamada,Tipo Respuesta,Detalle Respuesta,...,Franquicia,Ciudad,Zona,Barrio,Dirección,Teléfono verificado,Verificado por,Detalle Suscripcion,Source.Name,Fecha_Modificacion_Archivo
0,BJ1916,502434364,"FARMACIA FARMA BEJUMA, C.A.",ACTIVO,42.31,2026-04-01,09:31:00,GESTION OFICINA COMERCIAL,RECLAMO DEL SERVICIO,INTERMITENCIA,...,FIBEX BEJUMA,BEJUMA,BEJUMA,CENTRO,"AV FUNDADORES Y ANZOÁTEGUI SECT CENTRO # N/A, ...",NO VERIFICADO,None,PYME_60 MBPS,Data - ATC 01-04-2026 al 06-04-2026.xlsx,2026-04-06 11:43:36.364414
1,BJ2342,4462772,RAQUEL MARIA DIAZ DE HENRIQUEZ,POR RETIRAR,0,2026-04-06,10:01:00,GESTION OFICINA COMERCIAL,RETIRO DE SERVICIO,CONTRATÓ OTRO PROVEEDOR,...,FIBEX BEJUMA,BEJUMA,BEJUMA,CENTRO,"CL URDANETA # S/N, CENTRO",NO VERIFICADO,None,"BLACK FRIDAY FTTH_200 A 400, FIBEXPLAY HOME GR...",Data - ATC 01-04-2026 al 06-04-2026.xlsx,2026-04-06 11:43:36.364414
2,BL0003,16098705,FELIX RAMON MANZANILLA BRETO,ACTIVO,15,2026-04-01,15:43:00,GESTION OFICINA COMERCIAL,PAGO DEL SERVICIO,PUNTO DE VENTA,...,FIBEX BELEN,VALENCIA,BELEN,BELEN,"SECT CASCO BELEN AV BOLIVAR # 105, BELEN",NO VERIFICADO,None,"FIBEXPLAY HOME GRATIS, PLAN 2025 FTTH 600",Data - ATC 01-04-2026 al 06-04-2026.xlsx,2026-04-06 11:43:36.364414
3,BL0061,14025918,ERIKA DHAYANA GOMEZ FERNANDEZ,ACTIVO,0,2026-04-02,12:01:00,GESTION OFICINA COMERCIAL,PAGO DEL SERVICIO,PUNTO DE VENTA,...,FIBEX BELEN,VALENCIA,BELEN,BELEN,"SECT LOS BAñOS AV BOLIVAR # S/N, BELEN",NO VERIFICADO,None,"ASISTENCIA MEDICA INTEGRAL, FIBEXPLAY HOME GRA...",Data - ATC 01-04-2026 al 06-04-2026.xlsx,2026-04-06 11:43:36.364414
4,BL0068,16538367,CARLOS LUIS ASCANIO BALOA,ACTIVO,9.8,2026-04-03,08:52:00,GESTION OFICINA COMERCIAL,PAGO DEL SERVICIO,PAGO MOVIL,...,FIBEX BELEN,VALENCIA,BELEN,BELEN,"SECT JOSE ANTONIO ESAA CL 10 DE OCTUBRE # 09, ...",NO VERIFICADO,None,"FIBEXPLAY HOME GRATIS, PLAN 2025 FTTH 600",Data - ATC 01-04-2026 al 06-04-2026.xlsx,2026-04-06 11:43:36.364414
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64291,VC6801,22341518,JACLINE JOLEYDI FRANCO,ACTIVO,10,2026-04-13,14:42:00,GESTION OFICINA COMERCIAL,PAGO DEL SERVICIO,PAGO MOVIL,...,FIBEX VILLA DE CURA,VILLA DE CURA,VILLA DE CURA,MATA CAFÉ,"SECT LA PARAGUA 1 # 59-1, MATA CAFÉ",NO VERIFICADO,None,"CONECTADOS BASICO ASISTENCIA MEDICA 2.5$, CONE...",Data - ATC 13-04-2026 al 16-04-2026.xlsx,2026-04-16 08:24:00.773925
64292,YG2119,26840230,GABRIEL JOSUE DIAZ CABALLERO,ACTIVO,0,2026-04-13,13:27:00,GESTION OFICINA COMERCIAL,PAGO DEL SERVICIO,PAGO MOVIL,...,FIBEX YAGUA,VALENCIA,GUACARA,GUACARA,"URB SAMAN SECT SECTOR 3 # #33, GUACARA",NO VERIFICADO,None,"FIBEXPLAY HOME GRATIS, PLAN 2025 FTTH 250",Data - ATC 13-04-2026 al 16-04-2026.xlsx,2026-04-16 08:24:00.773925
64293,YG2534,30430630,MAIFIORLIC VICTORIA VéLIZ BRICEñO,ACTIVO,-0.17,2026-04-13,08:33:00,GESTION OFICINA COMERCIAL,RECLAMO DEL SERVICIO,SIN SERVICIO,...,FIBEX YAGUA,YAGUA,YAGUA,YAGUA,"CL AURORA SECT YAGUA # S/N, YAGUA",NO VERIFICADO,None,"300MB - PROMO ABRIL2025, FIBEXPLAY HOME GRATIS",Data - ATC 13-04-2026 al 16-04-2026.xlsx,2026-04-16 08:24:00.773925
64294,YG2912,26573816,ISMARY FRANYELI GOMEZ MATA,ACTIVO,0,2026-04-13,12:12:00,GESTION OFICINA COMERCIAL,PAGO DEL SERVICIO,PAGO MOVIL,...,FIBEX YAGUA,YAGUA,YAGUA,LOS MERECURES,"SECT LOS MERECURES CL 5 DE JULIO # S/N , LOS M...",NO VERIFICADO,None,"FIBEXPLAY HOME GRATIS, PLAN 2025 FTTH 1.5 G",Data - ATC 13-04-2026 al 16-04-2026.xlsx,2026-04-16 08:24:00.773925


In [25]:
import pandas as pd
df = pd.read_parquet(r"C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Afluencia_Gold.parquet")
display(df.columns)
col = ['Ciudad']
df = df[col]
df = df.drop_duplicates().reset_index(drop=True)
df.to_excel(r"C:\Users\josperez\Downloads\ciudades2.xlsx", index=False)
display(df)

Index(['N° Abonado', 'Documento', 'Estatus', 'Fecha', 'Vendedor',
       'Grupo Afinidad', 'Nombre Franquicia', 'Ciudad', 'Tipo de afluencia',
       'Oficina', 'Clasificacion', 'Hora', 'Mes', 'Estado_Sede', 'Tipo_Sede'],
      dtype='object')

,Ciudad
0,MARACAY
1,MATURIN
2,EL LIMóN
3,SANTA RITA
4,TURMERO
...,...
116,ROMULO GALLEGOS (LAS VEGAS)
117,MAGDALENO
118,BERMúDEZ (CARúPANO)
119,ACEVEDO (CAUCAGUA)


In [12]:
import pandas as pd
df_cobranza = pd.read_parquet(r"C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\bronze_data\Cobranza_Raw_Bronze.parquet")
display(df_cobranza.info())
df_cobranza['Fecha Llamada'] = pd.to_datetime(df_cobranza['Fecha Llamada'], errors='coerce')
mask_april = (df_cobranza['Fecha Llamada'] >= '2026-04-22') & (df_cobranza['Fecha Llamada'] <= '2026-04-22')
df_cobranza_april = df_cobranza[mask_april]
df_cobranza_april = df_cobranza_april.drop_duplicates(subset=['N° Abonado','Fecha Llamada','Hora Llamada',]).reset_index(drop=True)
        
display(df_cobranza_april)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 435289 entries, 0 to 435288
Data columns (total 25 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   N° Abonado                  435289 non-null  object        
 1   Cliente                     435289 non-null  object        
 2   Estatus                     435289 non-null  object        
 3   Saldo                       435289 non-null  object        
 4   Fecha Llamada               435289 non-null  object        
 5   Hora Llamada                435289 non-null  object        
 6   Tipo Llamada                435289 non-null  object        
 7   Tipo Respuesta              435289 non-null  object        
 8   Detalle Respuesta           435289 non-null  object        
 9   Responsable                 435289 non-null  object        
 10  Observación                 266590 non-null  object        
 11  Franquicia                  435289 non-

None

,N° Abonado,Cliente,Estatus,Saldo,Fecha Llamada,Hora Llamada,Tipo Llamada,Tipo Respuesta,Detalle Respuesta,Responsable,...,Source.Name,Documento,Suscripción,Grupo Afinidad,Ciudad,Teléfono,Zona,Dirección,Verificado por,Fecha_Modificacion_Archivo
0,BQ10466,JUAN VICENTE BRITO RIVERO,CORTADO,25,2026-04-22,15:06:00,GESTION COBRANZA OC,CLIENTE CORTADO,NO CONTESTA,ARALEX PIÑA OFIC DA VINCI BARQUISIMETO,...,Data - OCobranza 20-04-2026 al 23-04-2026.xlsx,11425230,25,HOGAR,BARQUISIMETO,4163582655,CENTRO,"SECT RUIZ PINEDA I # S/N, CENTRO OESTE",None,2026-04-23 08:35:09.693713
1,BQ11406,LUIS RAMON RIOS,CORTADO,25,2026-04-22,10:40:00,GESTION COBRANZA OC,CLIENTE CORTADO,INFORMADO PROMOCION,ARALEX PIÑA OFIC DA VINCI BARQUISIMETO,...,Data - OCobranza 20-04-2026 al 23-04-2026.xlsx,18054647,25,HOGAR,BARQUISIMETO,4125573267,CENTRO,SECT BARRIO EL BOLIVAR CL 10 CON CARRERA 4 SEC...,None,2026-04-23 08:35:09.693713
2,BQ11433,JOBELLY DEL CARMEN PARIS MARTINEZ,CORTADO,25,2026-04-22,10:24:00,GESTION COBRANZA OC,CLIENTE CORTADO,SE INFORMA DEUDA,ARALEX PIÑA OFIC DA VINCI BARQUISIMETO,...,Data - OCobranza 20-04-2026 al 23-04-2026.xlsx,13342763,25,HOGAR,BARQUISIMETO,4145341092,CENTRO,SECT BARRIO EL BOLIVAR CL 9 ENTRE CARRERAS 4 Y...,None,2026-04-23 08:35:09.693713
3,BQ11524,KARLA VALENTINA RAMOS GARCIA,CORTADO,28.83,2026-04-22,08:50:00,GESTION COBRANZA OC,CLIENTE CORTADO,NO CONTESTA,ARALEX PIÑA OFIC DA VINCI BARQUISIMETO,...,Data - OCobranza 20-04-2026 al 23-04-2026.xlsx,30014598,30,HOGAR,BARQUISIMETO,4245841791,CENTRO,SECT BARRIO EL BOLIVAR CL 4 ENTRE CARRERAS 8 Y...,None,2026-04-23 08:35:09.693713
4,BQ15504,MARSY GUADALUPE GARCIA ALVAREZ,CORTADO,27.06,2026-04-22,14:17:00,GESTION COBRANZA OC,CLIENTE CORTADO,NO CONTESTA,ARALEX PIÑA OFIC DA VINCI BARQUISIMETO,...,Data - OCobranza 20-04-2026 al 23-04-2026.xlsx,14648161,30,HOGAR,BARQUISIMETO,4261072541,ESTE METROPOLIS,SECT RUIZ PINEDA SAN ANTONIO AV PRINCIPAL # S...,None,2026-04-23 08:35:09.693713
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1447,V98228,BILLY JOSE VARGAS RIERA,CORTADO,25,2026-04-22,10:37:00,GESTION COBRANZA OC,CLIENTE CORTADO,NO CONTESTA,SORELYS HERRERA OFIC ALIANZA MALL,...,Data - OCobranza 20-04-2026 al 23-04-2026.xlsx,30096186,25,HOGAR,VALENCIA,4121309024,TOCUYITO,"SECT 5 CL AYACUCHO # SIN NúMERO , FUNDACION CAP",None,2026-04-23 08:35:09.693713
1448,V98282,LEOMARI CARBELIS PéREZ ESCALONA,CORTADO,45,2026-04-22,10:38:00,GESTION COBRANZA OC,CLIENTE CORTADO,NUMERO ERRADO,SORELYS HERRERA OFIC ALIANZA MALL,...,Data - OCobranza 20-04-2026 al 23-04-2026.xlsx,14271265,32.5,HOGAR,VALENCIA,4244262564,GUACARA,SECT ARAGUITA URB EZEQUIEL ZAMORA # CASA NúMER...,None,2026-04-23 08:35:09.693713
1449,V98295,MARIBEL MARGARITA GARCíA MORILLO,CORTADO,62,2026-04-22,10:39:00,GESTION COBRANZA OC,CLIENTE CORTADO,NO CONTESTA,SORELYS HERRERA OFIC ALIANZA MALL,...,Data - OCobranza 20-04-2026 al 23-04-2026.xlsx,7070254,30,HOGAR,VALENCIA,4144701261,LOS GUAYOS,"CLJ BOLIVAR # 16, CENTRO DE LOS GUAYOS",None,2026-04-23 08:35:09.693713
1450,V98302,ALI FERNANDO CAMPOS MARTINEZ,CORTADO,32,2026-04-22,10:57:00,GESTION COBRANZA OC,CLIENTE CORTADO,NO CONTESTA,SORELYS HERRERA OFIC ALIANZA MALL,...,Data - OCobranza 20-04-2026 al 23-04-2026.xlsx,5752051,32,HOGAR,VALENCIA,4241331104,GUACARA,SECT ARAGUITA URB EZEQUIEL ZAMORA # CASA NúMER...,None,2026-04-23 08:35:09.693713


In [11]:
import pandas as pd
df_cobranza = pd.read_parquet(r"C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\Llamadas_Cobranza_Gold.parquet")
display(df_cobranza.info())
df_cobranza['Fecha Llamada'] = pd.to_datetime(df_cobranza['Fecha Llamada'], errors='coerce')
mask_april = (df_cobranza['Fecha Llamada'] >= '2026-04-23') & (df_cobranza['Fecha Llamada'] <= '2026-04-23')
df_cobranza_april = df_cobranza[mask_april]
df_cobranza_april = df_cobranza_april.drop_duplicates(subset=['N° Abonado','Fecha Llamada','Hora',]).reset_index(drop=True)
        
display(df_cobranza_april)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354337 entries, 0 to 354336
Data columns (total 25 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   N° Abonado                  354337 non-null  object        
 1   Estatus                     354337 non-null  object        
 2   Saldo                       354337 non-null  object        
 3   Fecha Llamada               354337 non-null  object        
 4   Hora                        354337 non-null  object        
 5   Tipo Llamada                354337 non-null  object        
 6   Tipo Respuesta              354337 non-null  object        
 7   Detalle Respuesta           354337 non-null  object        
 8   Responsable                 354337 non-null  object        
 9   Observación                 224103 non-null  object        
 10  Franquicia                  354337 non-null  object        
 11  Barrio                      349051 non-

None

,N° Abonado,Estatus,Saldo,Fecha Llamada,Hora,Tipo Llamada,Tipo Respuesta,Detalle Respuesta,Responsable,Observación,...,Documento,Suscripción,Grupo Afinidad,Ciudad,Teléfono,Zona,Dirección,Verificado por,Fecha_Modificacion_Archivo,Canal
0,LCH16562,POR RETIRAR,40,2026-04-23,13:00,GESTION COBRANZA OC,CLIENTE CORTADO,ABANDONO EL SERVICIO POR FALLAS,JENNIFER GONZALEZ OFIC CLARINES,CLIENTE SOLICITO RETIRO DE EQUIPOS INDICA QUE ...,...,24831293,40,PYMES,CLARINES,4248401885,CLARINES,AV FERNANDEZ PADILLA SECT CASCO CENTRAL SUR # ...,None,2026-04-24 16:12:17.053005,OFICINA COMERCIAL
1,LCH39843,CORTADO,35,2026-04-23,13:00,GESTION COBRANZA OC,CLIENTE CORTADO,NO TIENE DINERO,JESUS RAFAEL ESCOBAR CORDOVA OFIC ANZOATEGI,None,...,8228998,35,HOGAR,BARCELONA,4148104321,BARCELONA,"SECT LOS UNIDOS CL 2 # 97, PUENTE AYALA",None,2026-04-24 16:12:17.053005,OFICINA COMERCIAL
2,LCH40803,CORTADO,30,2026-04-23,12:00,GESTION COBRANZA OC,CLIENTE CORTADO,NO CONTESTA,JESUS RAFAEL ESCOBAR CORDOVA OFIC ANZOATEGI,CLIENTE NO CONTESTA,...,20343736,30,HOGAR,BARCELONA,4148077986,BARCELONA,"CLJ LIBERTAD # 4, BARRIO UNIVERSITARIO",None,2026-04-24 16:12:17.053005,OFICINA COMERCIAL
3,LCH40872,CORTADO,35,2026-04-23,12:00,GESTION COBRANZA OC,CLIENTE CORTADO,NO CONTESTA,JESUS RAFAEL ESCOBAR CORDOVA OFIC ANZOATEGI,NO CONTESTA,...,22036286,37,HOGAR,CUMANA(BOLIVAR),4129059159,MARIGUITAR,"CL PRINCIPAL # 100, GOLINDANO",None,2026-04-24 16:12:17.053005,OFICINA COMERCIAL
4,LCH40912,CORTADO,30,2026-04-23,10:00,GESTION COBRANZA OC,CLIENTE CORTADO,PROMESA DE PAGO,JESUS RAFAEL ESCOBAR CORDOVA OFIC ANZOATEGI,INDICA QUE PAGA ANTES DEL 30,...,4616115,30,HOGAR,SAN ANTONIO,4140973070,CAPAYACUAR,"CL APARICIO # SN , SAN ANTONIO",None,2026-04-24 16:12:17.053005,OFICINA COMERCIAL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2016,BQ5325,CORTADO,44.85,2026-04-23,08:00,GESTION COBRANZA OC,CLIENTE CORTADO,NO CONTESTA,ARALEX PIÑA OFIC DA VINCI BARQUISIMETO,CLIENTE NO RESPONDE A LLAMADA DE COBRO.,...,19883918,25,PYMES,BARQUISIMETO,4120513479,OESTE,"SECT RUIZ PINEDA 2 VRD 11 # S/N, OESTE",None,2026-04-24 16:12:17.053005,OFICINA COMERCIAL
2017,BQ6201,ACTIVO,4.05,2026-04-23,09:00,GESTION COBRANZA OC,CLIENTE CORTADO,REACTIVA SERVICIO,ELIANNY PERAZA VARGAS OFIC BQTO,"CLIENTE EMITIÓ PAGO EL 21-01, PENDIENTE 4.01$",...,5939613,25,HOGAR,BARQUISIMETO,4125202733,CENTRO,SECT LA MUNICIPAL CL 5 ENTRE VEREDA 1 Y 2 # S...,None,2026-04-24 16:12:17.053005,OFICINA COMERCIAL
2018,BQ8810,CORTADO,23.68,2026-04-23,12:00,GESTION COBRANZA OC,CLIENTE CORTADO,NO CONTESTA,ARALEX PIÑA OFIC DA VINCI BARQUISIMETO,CLIENTE NO RESPONDE A LLAMADA DE COBRO.,...,27209622,25,HOGAR,BARQUISIMETO,4120935999,CENTRO,"SECT LOS LIBERTADORES CL SECTOR 3 # S/N, CEN...",None,2026-04-24 16:12:17.053005,OFICINA COMERCIAL
2019,C10724,CORTADO,25,2026-04-23,14:00,GESTION COBRANZA OC,CLIENTE CORTADO,SE INFORMA DEUDA,KARIANNYS RODRIGUEZ OFIC EL PARAISO CARACAS,"SE CONTACTA CLIENTE, INDICA QUE YA NO SE ENCUE...",...,26794851,25,HOGAR,CARACAS,4142691875,ANTIMANO,"AV AV INTERCOMUNAL # N/A, ANTIMANO",None,2026-04-24 16:12:17.053005,OFICINA COMERCIAL


In [17]:
import pandas as pd
df_ont = pd.read_parquet(r"C:\Users\josperez\Documents\A-DataStack\01-Proyectos\01-Data_PipelinesFibex\02_Data_Lake\gold_data\ONT_OFF_Gold.parquet")
display(df_ont.info())
display(df_ont.describe(include='all'))
display(df_ont)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1142 entries, 0 to 1141
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   N° Abonado         1142 non-null   object
 1   Documento          1142 non-null   object
 2   Cliente            1142 non-null   object
 3   Estatus            1142 non-null   object
 4   Fecha              1142 non-null   object
 5   Hora               1142 non-null   object
 6   Tipo Llamada       1142 non-null   object
 7   Tipo Respuesta     1142 non-null   object
 8   Detalle Respuesta  1142 non-null   object
 9   Vendedor           1142 non-null   object
 10  Suscripción        1142 non-null   object
 11  Grupo Afinidad     1142 non-null   object
 12  Nombre Franquicia  1142 non-null   object
 13  Ciudad             1142 non-null   object
dtypes: object(14)
memory usage: 125.0+ KB


None

,N° Abonado,Documento,Cliente,Estatus,Fecha,Hora,Tipo Llamada,Tipo Respuesta,Detalle Respuesta,Vendedor,Suscripción,Grupo Afinidad,Nombre Franquicia,Ciudad
count,1142,1142,1142,1142,1142,1142,1142,1142,1142,1142,1142,1142,1142,1142
unique,1079,1079,1079,4,2,214,2,1,16,33,30,5,35,68
top,E0538,3081825168,HOTSPOT FIBEX TEMPORAL,ACTIVO,2026-04-25,13:00,GESTION OFICINA COMERCIAL,CLIENTE ACTIVO CON ONT APAGADA,CLIENTE NO CONTESTA,ELIANNY PERAZA VARGAS OFIC BQTO,30,HOGAR,FIBEX VALENCIA,VALENCIA
freq,5,5,5,945,688,157,868,1142,632,121,345,1041,378,374


,N° Abonado,Documento,Cliente,Estatus,Fecha,Hora,Tipo Llamada,Tipo Respuesta,Detalle Respuesta,Vendedor,Suscripción,Grupo Afinidad,Nombre Franquicia,Ciudad
0,T0442,26843446,EGLIMAR ALEXANDRA PINTO ARIAS,ACTIVO,2026-04-27,10:40:00,GESTIÓN CALL CENTER,CLIENTE ACTIVO CON ONT APAGADA,CLIENTE NO CONTESTA,DERICK LOPEZ ANALISTA CALL CENTER,25,HOGAR,FIBEX TINAQUILLO,TINAQUILLO
1,T13268,23436120,DISGLEIDY DEL CARMEN RODRIGUEZ MORENO,ACTIVO,2026-04-27,12:58:00,GESTIÓN CALL CENTER,CLIENTE ACTIVO CON ONT APAGADA,CLIENTE NO CONTESTA,DERICK LOPEZ ANALISTA CALL CENTER,23.2,HOGAR,FIBEX TINAQUILLO,TINAQUILLO
2,VC2054,10341923,OSCAR GREGORIO PEREIRA,SUSPENDIDO,2026-04-27,09:41:00,GESTIÓN CALL CENTER,CLIENTE ACTIVO CON ONT APAGADA,NO USA INTERNET EN ESE MOMENTO / ESTá DE VIAJE,VALESKA YAJAIRA PERALTA GONZALEZ ANALISTA CALL...,30,HOGAR,FIBEX VILLA DE CURA,VILLA DE CURA
3,C13807,14003808,EDWARD IGNACIO GOMEZ ALDANA,ACTIVO,2026-04-25,10:53:00,GESTIÓN CALL CENTER,CLIENTE ACTIVO CON ONT APAGADA,RECLAMO PENDIENTE SIN RESOLVER,MOISES WILFREDIS LOPEZ MERCADO ANALISTA DE CAL...,35,HOGAR,FIBEX CARACAS,CARACAS
4,C13913,3024344,MARIA CARPIO DE DUGARTE,ACTIVO,2026-04-25,10:56:00,GESTIÓN CALL CENTER,CLIENTE ACTIVO CON ONT APAGADA,CLIENTE NO CONTESTA,MOISES WILFREDIS LOPEZ MERCADO ANALISTA DE CAL...,30,HOGAR,FIBEX CARACAS,CARACAS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1137,V89085,24458274,LUIS EDUARDO MORALES GUARENAS,ACTIVO,2026-04-27,12:00,GESTION OFICINA COMERCIAL,CLIENTE ACTIVO CON ONT APAGADA,DIFICULTAD DE PAGO PENDIENTE,NAYIS TERESEN OFIC METROPOLIS VALENCIA,30,HOGAR,FIBEX VALENCIA,VALENCIA
1138,V92283,9442130,IRIS MILAGRO PEREZ,ACTIVO,2026-04-27,09:00,GESTION OFICINA COMERCIAL,CLIENTE ACTIVO CON ONT APAGADA,CLIENTE NO CONTESTA,YOLFRAN POLANCO OFIC METROPOLIS VALENCIA,32,HOGAR,FIBEX VALENCIA,SAN DIEGO
1139,V92386,23813708,YONATHAN MIGUEL PEñA LOPEZ,ACTIVO,2026-04-27,10:00,GESTION OFICINA COMERCIAL,CLIENTE ACTIVO CON ONT APAGADA,CLIENTE NO CONTESTA,MARIA GUZMAN OFIC METROPOLIS VALENCIA,30,HOGAR,FIBEX VALENCIA,VALENCIA
1140,V9445,20023738,ROSSY SOLEDAD DUQUE MOROCOIMA,ACTIVO,2026-04-27,12:00,GESTION OFICINA COMERCIAL,CLIENTE ACTIVO CON ONT APAGADA,EL EQUIPO PRESENTA FALLA O NO ENCIENDE,NAYIS TERESEN OFIC METROPOLIS VALENCIA,42,HOGAR,FIBEX VALENCIA,VALENCIA
